# GPR151 vs Opioid Receptors: Two-Phase Membrane Notebook

**Phase 1:** 6 systems Ã— 20 ns explicit membrane = screening (~6 hours on 2Ã— T4)
**Phase 2:** Re-run top hits Ã— 100 ns = validation (~10 hours on 2Ã— T4)

This notebook defaults to **explicit membrane** (POPC:POPE:CHL1).
Phase 1 runs automatically. After analysis, manually set `HIT_PAIRS` and run Phase 2.
If a pair survives 20 ns, it is worth extending. If it drifts apart, it is likely not a stable heterodimer.


## SECTION 0: Environment Setup


In [ ]:
# 0a. pip-install (OpenMM will be installed via conda-forge in cell 0b)
import sys, subprocess, time, os, shutil
t0 = time.time()
print('Installing for:', sys.executable)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pdbfixer', 'parmed', 'mdanalysis', 'biopython', 'rdkit', 'biotite',
    'matplotlib', 'pandas', 'numpy', 'scipy'], check=False)
print(f'pip done in {time.time()-t0:.0f}s')
if shutil.which('obabel') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    subprocess.run(['apt-get', 'install', '-y', '-q', 'openbabel'], check=True)
    print('obabel installed via apt-get')
else:
    print('obabel already available')


In [ ]:
# 0b. Miniconda + AmberTools + OpenMM (CUDA via conda-forge)
import os, subprocess, time, sys, glob, shutil
CONDA_PREFIX = '/opt/miniconda'
AMBER_ENV = f'{CONDA_PREFIX}/envs/ambertools'
OPENMM_ENV = f'{CONDA_PREFIX}/envs/openmm_env'
PY_VER = f'{sys.version_info.major}.{sys.version_info.minor}'
print(f'Python: {sys.executable}')
print(f'Python version: {PY_VER}')

# 1. Install Miniconda if not present
if not os.path.exists(f'{CONDA_PREFIX}/bin/conda'):
    t0 = time.time()
    subprocess.run(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
                    '-O', '/tmp/miniconda.sh'], check=True)
    subprocess.run(['bash', '/tmp/miniconda.sh', '-b', '-f', '-p', CONDA_PREFIX], check=True)
    os.remove('/tmp/miniconda.sh')
    print(f'Miniconda installed in {time.time()-t0:.0f}s')

conda_bin = f'{CONDA_PREFIX}/bin/conda'

# 2. Install AmberTools if not present
if not os.path.exists(f'{AMBER_ENV}/bin/packmol-memgen'):
    t0 = time.time()
    subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel',
                    'https://repo.anaconda.com/pkgs/main'], check=False)
    subprocess.run([conda_bin, 'tos', 'accept', '--override-channels', '--channel',
                    'https://repo.anaconda.com/pkgs/r'], check=False)
    subprocess.run([conda_bin, 'create', '-y', '-q', '-n', 'ambertools',
                    '-c', 'conda-forge', '-c', 'ambermd', 'ambertools'], check=True)
    subprocess.run([conda_bin, 'clean', '-a', '-y'], check=True)
    print(f'AmberTools built in {time.time()-t0:.0f}s')
else:
    print('AmberTools already present')

# 3. Install OpenMM with CUDA support via conda-forge
if not os.path.exists(f'{OPENMM_ENV}/bin/python'):
    t0 = time.time()
    subprocess.run([conda_bin, 'create', '-y', '-q', '-n', 'openmm_env',
                    'python=3.12', '-c', 'conda-forge'], check=True)
    subprocess.run([conda_bin, 'run', '-n', 'openmm_env',
                    'conda', 'install', '-y', '-q', '-c', 'conda-forge',
                    'openmm', 'cudatoolkit'], check=True)
    print(f'OpenMM (CUDA) built in {time.time()-t0:.0f}s')
else:
    print('OpenMM env already present')

PACKMOL_MEMGEN = f'{AMBER_ENV}/bin/packmol-memgen'
TLEAP = f'{AMBER_ENV}/bin/tleap'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
assert os.path.exists(PACKMOL_MEMGEN), 'packmol-memgen not found'
assert os.path.exists(TLEAP), 'tleap not found'
assert os.path.exists(OPENMM_PYTHON), 'openmm_env python not found'
print('AmberTools ready')
print('OpenMM env:', OPENMM_ENV)


In [ ]:
# 0c. Write the MD engine script to disk (runs via conda Python with CUDA OpenMM)
import os, textwrap
WORK_DIR = '/kaggle/working/gpr151_opioid_pipeline'
os.makedirs(WORK_DIR, exist_ok=True)
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'

MD_ENGINE = textwrap.dedent(r"""#!/usr/bin/env python3
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

# Fix libstdc++ ABI mismatch between system GCC and conda-forge GCC
os.environ['LD_LIBRARY_PATH'] = (
    f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:'
    + os.environ.get('LD_LIBRARY_PATH', '')
)

import openmm
import openmm.unit as unit
from openmm import Platform, LangevinMiddleIntegrator, CustomExternalForce
from openmm.app import PME, HBonds, AmberPrmtopFile, AmberInpcrdFile, PDBFile, Simulation
from openmm.app import DCDReporter, StateDataReporter, CheckpointReporter
from openmm.app import OBC2, NoCutoff

CONFIG_PATH = sys.argv[1]
with open(CONFIG_PATH) as f:
    cfg = json.load(f)
PHYSICS_MODE = cfg['PHYSICS_MODE']
STEP_SIZE = cfg['STEP_SIZE']
USE_HMASS = cfg['USE_HMASS']
CHECKPOINT_STEPS = cfg['CHECKPOINT_STEPS']
N_STEPS_TOTAL = cfg['N_STEPS_TOTAL']
stage = sys.argv[2]
pair_name = sys.argv[3]
work_dir = sys.argv[4]
gpu_idx = int(sys.argv[5])

def get_platform(gpu_idx):
    platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
    print(f'[{pair_name}] Available platforms: {platforms}')
    if 'CUDA' in platforms:
        print(f'[{pair_name}] Using CUDA platform on GPU {gpu_idx}')
        return Platform.getPlatformByName('CUDA'), {'CudaDeviceIndex': str(gpu_idx), 'CudaPrecision': 'mixed'}
    elif 'OpenCL' in platforms:
        print(f'[{pair_name}] Using OpenCL platform on GPU {gpu_idx}')
        return Platform.getPlatformByName('OpenCL'), {'OpenCLDeviceIndex': str(gpu_idx)}
    else:
        print(f'[{pair_name}] WARNING: No GPU platform available. Using CPU.')
        return Platform.getPlatformByName('CPU'), {}

def run_equil():
    prmtop = AmberPrmtopFile(f'{work_dir}/{pair_name}.prmtop')
    inpcrd = AmberInpcrdFile(f'{work_dir}/{pair_name}.inpcrd')
    if PHYSICS_MODE == 'explicit':
        system = prmtop.createSystem(nonbondedMethod=PME, nonbondedCutoff=1.0*unit.nanometer,
                                     constraints=HBonds, ewaldErrorTolerance=0.0005)
    else:
        system = prmtop.createSystem(implicitSolvent=OBC2, nonbondedMethod=NoCutoff, constraints=HBonds)
    if USE_HMASS:
        for i in range(system.getNumParticles()):
            if system.getParticleMass(i) == 1.008 * unit.amu:
                system.setParticleMass(i, 4.0 * unit.amu)
    force = CustomExternalForce('0.5*k*((x-x0)^2+(y-y0)^2+(z-z0)^2)')
    force.addGlobalParameter('k', 4184.0)
    for p in ['x0','y0','z0']: force.addPerParticleParameter(p)
    for atom in prmtop.topology.atoms():
        if atom.element.symbol != 'H':
            pos = inpcrd.positions[atom.index].value_in_unit(unit.nanometer)
            force.addParticle(atom.index, pos)
    system.addForce(force)
    integrator = LangevinMiddleIntegrator(300*unit.kelvin, 1.0/unit.picosecond, STEP_SIZE*unit.picosecond)
    platform, props = get_platform(gpu_idx)
    simulation = Simulation(prmtop.topology, system, integrator, platform, props)
    simulation.context.setPositions(inpcrd.positions)
    print(f'{pair_name}: Minimizing...')
    simulation.minimizeEnergy(maxIterations=5000)
    print(f'{pair_name}: Heating to 310K...')
    simulation.context.setVelocitiesToTemperature(100*unit.kelvin)
    for temp in [100, 200, 310]:
        integrator.setTemperature(temp*unit.kelvin)
        simulation.step(5000)
    print(f'{pair_name}: Equilibration (10 ns)...')
    simulation.reporters.append(DCDReporter(f'{work_dir}/{pair_name}_equil.dcd', 5000))
    simulation.reporters.append(StateDataReporter(sys.stdout, 5000, step=True,
        potentialEnergy=True, temperature=True, volume=True, speed=True, elapsedTime=True))
    simulation.step(2500000)
    simulation.saveState(f'{work_dir}/{pair_name}_equil.xml')
    positions = simulation.context.getState(getPositions=True).getPositions()
    with open(f'{work_dir}/{pair_name}_equil.pdb', 'w') as f:
        PDBFile.writeFile(prmtop.topology, positions, f)
    print(f'{pair_name}: Equilibration complete')

def run_prod():
    prmtop = AmberPrmtopFile(f'{work_dir}/{pair_name}.prmtop')
    equil_pdb = f'{work_dir}/{pair_name}_equil.pdb'
    if os.path.exists(equil_pdb):
        pdb = PDBFile(equil_pdb)
    else:
        pdb = AmberInpcrdFile(f'{work_dir}/{pair_name}.inpcrd')
    if PHYSICS_MODE == 'explicit':
        system = prmtop.createSystem(nonbondedMethod=PME, nonbondedCutoff=1.0*unit.nanometer,
                                     constraints=HBonds, ewaldErrorTolerance=0.0005)
    else:
        system = prmtop.createSystem(implicitSolvent=OBC2, nonbondedMethod=NoCutoff, constraints=HBonds)
    if USE_HMASS:
        for i in range(system.getNumParticles()):
            if system.getParticleMass(i) == 1.008 * unit.amu:
                system.setParticleMass(i, 4.0 * unit.amu)
    integrator = LangevinMiddleIntegrator(310*unit.kelvin, 1.0/unit.picosecond, STEP_SIZE*unit.picosecond)
    platform, props = get_platform(gpu_idx)
    simulation = Simulation(prmtop.topology, system, integrator, platform, props)
    dcd_path = f'{work_dir}/{pair_name}_prod.dcd'
    chk_file = f'{work_dir}/{pair_name}_prod.chk'
    if os.path.exists(chk_file):
        simulation.loadCheckpoint(chk_file)
        state = simulation.context.getState()
        completed_steps = int(state.getTime().value_in_unit(unit.picosecond) / STEP_SIZE)
        print(f'[{pair_name} GPU{gpu_idx}]: resumed from checkpoint at {completed_steps} steps ({completed_steps*STEP_SIZE/1000:.1f} ns)')
    else:
        simulation.context.setPositions(pdb.positions)
        simulation.context.setVelocitiesToTemperature(310*unit.kelvin)
        completed_steps = 0
        print(f'[{pair_name} GPU{gpu_idx}]: starting fresh production')
    simulation.reporters.append(DCDReporter(dcd_path, 25000, append=os.path.exists(dcd_path)))
    log_file = f'{work_dir}/{pair_name}_prod.log'
    simulation.reporters.append(StateDataReporter(log_file, 25000, step=True,
        potentialEnergy=True, temperature=True, speed=True, elapsedTime=True, append=os.path.exists(log_file)))
    simulation.reporters.append(CheckpointReporter(chk_file, CHECKPOINT_STEPS))
    remaining = N_STEPS_TOTAL - completed_steps
    if remaining <= 0:
        print(f'[{pair_name} GPU{gpu_idx}]: already complete')
        return
    print(f'[{pair_name} GPU{gpu_idx}]: running {remaining} steps ({remaining*STEP_SIZE/1000:.1f} ns)')
    try:
        simulation.step(remaining)
        print(f'[{pair_name} GPU{gpu_idx}]: production complete')
    except Exception as e:
        print(f'[{pair_name} GPU{gpu_idx}]: ERROR - {e}')
        import traceback
        traceback.print_exc()

if __name__ == '__main__':
    if stage == 'equil':
        run_equil()
    elif stage == 'prod':
        run_prod()
    else:
        print(f'Unknown stage: {stage}')
        sys.exit(1)
""").strip()

md_path = f'{WORK_DIR}/md_engine.py'
with open(md_path, 'w') as f:
    f.write(MD_ENGINE)
print(f'MD engine written to: {md_path}')


In [ ]:
# 0d. GPU check, working dirs, TIME BUDGET, and PHYSICS_MODE
import subprocess, shutil, os, time, glob, json
if shutil.which('nvidia-smi'):
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
    print('GPU:', r.stdout.strip())
WORK_DIR = '/kaggle/working/gpr151_opioid_pipeline'
os.makedirs(WORK_DIR, exist_ok=True)
PHYSICS_MODE = 'implicit'
TIME_BUDGET_HOURS = 10
NS_PER_SYSTEM = 20
USE_HMASS = True
STEP_SIZE = 0.004
CHECKPOINT_NS = 5
CHECKPOINT_STEPS = int(CHECKPOINT_NS * 1000 / STEP_SIZE)
N_STEPS_TOTAL = int(NS_PER_SYSTEM * 1000 / STEP_SIZE)
print(f'PHYSICS_MODE: {PHYSICS_MODE}')
print(f'NS_PER_SYSTEM: {NS_PER_SYSTEM} ns')
print(f'STEP_SIZE: {STEP_SIZE} ps ({STEP_SIZE*1000} fs)')
if PHYSICS_MODE == 'explicit':
    ns_per_hour_per_gpu = 10
    total_hours = (6 * NS_PER_SYSTEM) / (ns_per_hour_per_gpu * 2)
    print(f'Phase 1 (screening): {total_hours:.1f} hours on 2x T4')
    print('Phase 2 (hits -> 100 ns): ~10 hours for 2 hits on 2x T4')
    print('Total: ~16 hours across 2 sessions')
else:
    ns_per_hour_per_gpu = 65
    total_hours = (6 * NS_PER_SYSTEM) / (ns_per_hour_per_gpu * 2)
    print(f'Estimated time (implicit): {total_hours:.1f} hours on 2x T4')
RUN_STAGE = {'setup':True, 'structures':True, 'truncation':True, 'complex_pred':True,
             'membrane':True, 'equil':True, 'production':True, 'analysis':True}
config = {
    'PHYSICS_MODE': PHYSICS_MODE,
    'STEP_SIZE': STEP_SIZE,
    'USE_HMASS': USE_HMASS,
    'CHECKPOINT_STEPS': CHECKPOINT_STEPS,
    'N_STEPS_TOTAL': N_STEPS_TOTAL,
}
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f)
print(f'Config saved to: {CONFIG_PATH}')

# Diagnostic: run in CONDA Python with explicit LD_LIBRARY_PATH
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
diag_script = "import openmm; from openmm import Platform; p=[Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]; print(p)"
import os
diag_env = os.environ.copy()
diag_env['LD_LIBRARY_PATH'] = f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + diag_env.get('LD_LIBRARY_PATH', '')
r = subprocess.run([OPENMM_PYTHON, '-c', diag_script], capture_output=True, text=True, env=diag_env)
print(f'OpenMM platforms (conda): {r.stdout.strip()}')
if r.returncode != 0:
    print(f'ERROR: {r.stderr.strip()}')
elif 'CUDA' in r.stdout:
    print('SUCCESS: CUDA platform is available via conda OpenMM')
else:
    print('WARNING: CUDA not found. Check conda env and GPU drivers.')

RESUME_INPUT = '/kaggle/input/gpr151-opioid-checkpoint'
if os.path.isdir(RESUME_INPUT):
    copied = 0
    for f in glob.glob(f'{RESUME_INPUT}/**/*', recursive=True):
        if os.path.isfile(f):
            dest = f.replace(RESUME_INPUT, WORK_DIR)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy2(f, dest)
            copied += 1
    print(f'Restored {copied} files from checkpoint')
else:
    print('No checkpoint - starting fresh')


## SECTION 1: Fetch AlphaFold Structures


In [ ]:
# 1a. Download AlphaFold structures
import requests, json
targets = {'gpr151':'Q8TDV0', 'gpr88':'Q9GZN0', 'mor':'P35372', 'kor':'P41145', 'dor':'P41143'}
for name, uniprot in targets.items():
    pdb_url = f'https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-model_v6.pdb'
    r = requests.get(pdb_url, timeout=60)
    r.raise_for_status()
    with open(f'{WORK_DIR}/{name}_alphafold.pdb', 'w') as f:
        f.write(r.text)
    ca = [l for l in r.text.split('\n') if l.startswith('ATOM') and l[12:16].strip()=='CA']
    print(f'{name}: {len(ca)} residues saved')
    r2 = requests.get(f'https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-predicted_aligned_error_v6.json', timeout=60)
    if r2.ok:
        with open(f'{WORK_DIR}/{name}_pae.json', 'w') as f:
            json.dump(r2.json(), f)


In [ ]:
# 1b. Fetch UniProt sequences and TM annotations
import requests
uniprot_data = {}
for name, uniprot in targets.items():
    r = requests.get(f'https://rest.uniprot.org/uniprotkb/{uniprot}.json', timeout=30)
    r.raise_for_status()
    data = r.json()
    uniprot_data[name] = data
    tms = [(f['location']['start']['value'], f['location']['end']['value'])
           for f in data.get('features',[]) if f.get('type')=='Transmembrane']
    data['tm_regions'] = tms
    print(f"{name}: {len(tms)} TMs, gene={data['genes'][0]['geneName']['value']}")


## SECTION 2: N-Terminal Truncation & Preparation


In [ ]:
# 2a. N-terminal truncation
from Bio import PDB
from Bio.PDB import PDBParser, PDBIO, Select
import warnings
warnings.filterwarnings('ignore')
class NTermTruncationSelect(Select):
    def __init__(self, start): self.start = start
    def accept_residue(self, r): return r.get_id()[1] >= self.start
    def accept_atom(self, a): return True
truncation = {}
for name in targets:
    tms = uniprot_data[name].get('tm_regions', [])
    truncation[name] = tms[0][0] if tms else {'gpr151':32, 'gpr88':36, 'mor':65, 'kor':59, 'dor':59}.get(name, 50)
    print(f"{name}: truncating at residue {truncation[name]}")
for name, start in truncation.items():
    s = PDBParser(QUIET=True).get_structure(name, f'{WORK_DIR}/{name}_alphafold.pdb')
    kept = [r for r in s.get_residues() if r.get_id()[1] >= start]
    removed = [r for r in s.get_residues() if r.get_id()[1] < start]
    out = f'{WORK_DIR}/{name}_delta_n.pdb'
    io = PDBIO(); io.set_structure(s); io.save(out, NTermTruncationSelect(start))
    print(f"{name}: {len(kept)} kept, {len(removed)} removed")


In [ ]:
# 2b. PDBFixer (write PDB manually without OpenMM import in notebook)
from pdbfixer import PDBFixer
def write_pdb_from_fixer(fixer, filename):
    with open(filename, 'w') as f:
        f.write("REMARK   1 PDBFixer prepared structure\n")
        atom_index = 1
        for chain in fixer.topology.chains():
            chain_id = chain.id if chain.id else 'A'
            for residue in chain.residues():
                res_name = residue.name[:3]
                try:
                    res_id = int(residue.id)
                except (ValueError, TypeError):
                    res_id = 1
                for atom in residue.atoms():
                    pos = fixer.positions[atom.index]
                    x, y, z = pos.x * 10, pos.y * 10, pos.z * 10
                    if atom.element and atom.element.symbol:
                        element = atom.element.symbol
                    else:
                        name = atom.name.strip()
                        if name.startswith('H'): element = 'H'
                        elif name.startswith('C'): element = 'C'
                        elif name.startswith('N'): element = 'N'
                        elif name.startswith('O'): element = 'O'
                        elif name.startswith('S'): element = 'S'
                        elif name.startswith('P'): element = 'P'
                        elif name.startswith('F'): element = 'F'
                        elif name.startswith('Cl') or name.startswith('CL'): element = 'Cl'
                        elif name.startswith('Br') or name.startswith('BR'): element = 'Br'
                        elif name.startswith('I'): element = 'I'
                        elif name.startswith('Mg') or name.startswith('MG'): element = 'Mg'
                        elif name.startswith('Ca') or name.startswith('CA'): element = 'Ca'
                        elif name.startswith('Na') or name.startswith('NA'): element = 'Na'
                        elif name.startswith('K'): element = 'K'
                        elif name.startswith('Zn') or name.startswith('ZN'): element = 'Zn'
                        elif name.startswith('Fe') or name.startswith('FE'): element = 'Fe'
                        else: element = 'X'
                    f.write(f"ATOM  {atom_index:5d} {atom.name:<4s} {res_name:3s} {chain_id:1s}{res_id:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {element:>2s}\n")
                    atom_index += 1
        f.write("END\n")
for name in targets:
    fixer = PDBFixer(filename=f'{WORK_DIR}/{name}_delta_n.pdb')
    fixer.findMissingResidues(); fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues(); fixer.removeHeterogens(keepWater=False)
    fixer.findMissingAtoms(); fixer.addMissingAtoms()
    write_pdb_from_fixer(fixer, f'{WORK_DIR}/{name}_prepared.pdb')
    print(f"{name}: {fixer.topology.getNumResidues()} residues, {fixer.topology.getNumAtoms()} atoms")


In [ ]:
# 2c. Convert to PDBQT
import os
for name in targets:
    ret = os.system(f"obabel {WORK_DIR}/{name}_prepared.pdb -O {WORK_DIR}/{name}_prepared.pdbqt -xr -p 7.4")
    sz = os.path.getsize(f'{WORK_DIR}/{name}_prepared.pdbqt')
    print(f"{name}: {'OK' if ret==0 and sz>0 else 'FAILED'} ({sz:,} bytes)")


## SECTION 3: Complex Structure Prediction (Geometry Scan)


In [ ]:
# 3a. Load structures and align principal axis to z
import numpy as np
from biotite.structure.io import load_structure
from biotite.structure import filter_amino_acids
def align_to_z(arr):
    ca = arr[arr.atom_name == 'CA']
    com = ca.coord.mean(axis=0)
    centered = arr.coord - com
    cov = np.cov(centered[arr.atom_name == 'CA'].T)
    e, v = np.linalg.eigh(cov)
    principal = v[:, np.argmax(e)]
    z = np.array([0.,0.,1.])
    c = np.dot(principal, z)
    if abs(c) > 0.999:
        arr.coord = centered
        return arr
    vcross = np.cross(principal, z)
    s = np.linalg.norm(vcross)
    vx = np.array([[0,-vcross[2],vcross[1]],[vcross[2],0,-vcross[0]],[-vcross[1],vcross[0],0]])
    R = np.eye(3) + vx + vx@vx*((1-c)/(s**2))
    arr.coord = (R @ centered.T).T
    return arr
structures = {}
for name in targets:
    s = load_structure(f'{WORK_DIR}/{name}_prepared.pdb')
    s = s[filter_amino_acids(s)]
    align_to_z(s)
    structures[name] = s
    print(f'{name}: loaded {len(s)} atoms, aligned to z')


In [ ]:
# 3b. Geometry scan
def score_contacts(ref, mobile, cutoff=5.0):
    dists = np.sqrt(np.sum((ref.coord[:,None] - mobile.coord[None,:])**2, axis=2))
    contacts = np.sum(dists < cutoff)
    hydrophobic = {'ALA','VAL','LEU','ILE','MET','PHE','TRP','PRO'}
    def _res_name_str(n):
        return n.decode() if isinstance(n, bytes) else str(n)
    ref_h = np.array([_res_name_str(n) in hydrophobic for n in ref.res_name])
    mob_h = np.array([_res_name_str(n) in hydrophobic for n in mobile.res_name])
    h_contacts = np.sum((dists < cutoff) & ref_h[:,None] & mob_h[None,:])
    return contacts, h_contacts
def scan_interface(ref, mobile, distance=40.0, step_deg=15):
    best_score, best_coords, best_angle = -1, None, 0
    for angle in range(0, 360, step_deg):
        theta = np.radians(angle)
        rot = np.array([[np.cos(theta),-np.sin(theta),0],[np.sin(theta),np.cos(theta),0],[0,0,1]])
        m = mobile.copy()
        m.coord = (rot @ m.coord.T).T
        m.coord[:,0] += distance
        contacts, h_contacts = score_contacts(ref, m)
        score = contacts + 2*h_contacts
        if score > best_score:
            best_score, best_coords, best_angle = score, m.coord.copy(), angle
    return best_angle, best_score, best_coords
orphan_list = ['gpr151', 'gpr88']
opioid_list = ['mor', 'kor', 'dor']
complexes = {}
for orphan in orphan_list:
    for opioid in opioid_list:
        pair = f'{orphan}_{opioid}'
        angle, score, coords = scan_interface(structures[orphan], structures[opioid])
        complexes[pair] = {'angle': angle, 'score': score, 'coords': coords}
        print(f'{pair}: best angle={angle} deg, score={score}')


In [ ]:
# 3c. Save best complex poses as PDB with chain IDs A/B
import json
import biotite.structure as struc
from biotite.structure.io.pdb import PDBFile as BiotitePDBFile
import numpy as np
for orphan in orphan_list:
    for opioid in opioid_list:
        pair = f'{orphan}_{opioid}'
        ref = structures[orphan]
        mob = structures[opioid].copy()
        mob.coord = complexes[pair]['coords']
        comp = struc.concatenate([ref, mob])
        n_ref = len(ref)
        comp.chain_id = np.array(['A']*n_ref + ['B']*(len(comp)-n_ref), dtype='U1')
        out = f'{WORK_DIR}/{pair}_complex.pdb'
        f = BiotitePDBFile(); f.set_structure(comp); f.write(out)
        n_A_res = len(np.unique(ref.res_id))
        n_B_res = len(np.unique(mob.res_id))
        with open(f'{WORK_DIR}/{pair}_split.json', 'w') as jf:
            json.dump({'n_A': int(n_A_res), 'n_B': int(n_B_res)}, jf)
        print(f'Saved: {out}')


## SECTION 4: System Preparation


In [ ]:
# 4a. Branch based on PHYSICS_MODE
print(f'PHYSICS_MODE = {PHYSICS_MODE}')
if PHYSICS_MODE == 'explicit':
    import os, subprocess
    os.chdir(WORK_DIR)
    amber_env = os.environ.copy()
    amber_env['AMBERHOME'] = AMBER_ENV
    amber_env['PATH'] = f'{AMBER_ENV}/bin:' + amber_env.get('PATH','')
    amber_env['LD_LIBRARY_PATH'] = f'{AMBER_ENV}/lib:' + amber_env.get('LD_LIBRARY_PATH','')
    for orphan in orphan_list:
        for opioid in opioid_list:
            pair = f'{orphan}_{opioid}'
            cmd = [PACKMOL_MEMGEN,
                   '--pdb', f'{pair}_complex.pdb',
                   '--lipids', 'POPC:POPE:CHL1', '--ratio', '2:2:1',
                   '--distxy_fix', '80', '--dist', '20', '--dist_wat', '20',
                   '--nloop', '15', '--nloop_all', '80', '--notprotonate']
            print(f'\nEmbedding {pair}...')
            r = subprocess.run(cmd, capture_output=True, text=True, env=amber_env)
            if r.returncode != 0:
                print(f'  FAILED: {r.stderr[-300:]}')
            else:
                print(f'  Success')
else:
    print('Implicit mode: skipping membrane embedding.')


In [ ]:
# 4b. tleap parameterization
import os, subprocess
os.chdir(WORK_DIR)
for orphan in orphan_list:
    for opioid in opioid_list:
        pair = f'{orphan}_{opioid}'
        if PHYSICS_MODE == 'explicit':
            bilayer = f'bilayer_{pair}_complex.pdb'
            if not os.path.exists(bilayer):
                print(f'{pair}: bilayer not found, skipping')
                continue
            leap_script = f'source leaprc.protein.ff19SB\nsource leaprc.lipid21\nsource leaprc.water.tip3p\n'
            leap_script += f'mol = loadpdb {bilayer}\n'
            leap_script += f'solvateBox mol TIP3PBOX 10.0\n'
            leap_script += f'addIons mol Na+ 0\naddIons mol Cl- 0\n'
        else:
            leap_script = f'source leaprc.protein.ff19SB\n'
            leap_script += f'set default PBRadii mbondi3\n'
            leap_script += f'mol = loadpdb {pair}_complex.pdb\n'
        leap_script += f'saveamberparm mol {pair}.prmtop {pair}.inpcrd\n'
        leap_script += f'savepdb mol {pair}_solvated.pdb\nquit'
        with open(f'{pair}_tleap.in', 'w') as f:
            f.write(leap_script)
        env = os.environ.copy(); env['AMBERHOME'] = AMBER_ENV
        r = subprocess.run([TLEAP, '-f', f'{pair}_tleap.in'], capture_output=True, text=True, env=env)
        if r.returncode == 0:
            print(f'{pair}: prmtop/inpcrd created')
        else:
            print(f'{pair}: tleap FAILED')


## SECTION 5: Equilibration MD


In [ ]:
# 5. Run equilibration for all 6 pairs via subprocess
import subprocess, os
MD_ENGINE = f'{WORK_DIR}/md_engine.py'
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
if RUN_STAGE['equil']:
    gpu_idx = 0
    for orphan in orphan_list:
        for opioid in opioid_list:
            pair = f'{orphan}_{opioid}'
            if os.path.exists(f'{WORK_DIR}/{pair}.prmtop'):
                print(f'\n--- Running equilibration for {pair} on GPU {gpu_idx} ---')
                r = subprocess.run([sys.executable, MD_ENGINE, CONFIG_PATH, 'equil', pair, WORK_DIR, str(gpu_idx)],
                                   capture_output=False, text=True)
                if r.returncode != 0:
                    print(f'ERROR: equilibration for {pair} failed')
                gpu_idx = 1 - gpu_idx
            else:
                print(f'{pair}: missing prmtop, skipping')
else:
    print('Skipping equilibration')


## SECTION 6: Production MD (Concurrent on 2 GPUs)


In [ ]:
# 6. Run production CONCURRENTLY on 2 GPUs via subprocess
import threading, subprocess, os
MD_ENGINE = f'{WORK_DIR}/md_engine.py'
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
if RUN_STAGE['production']:
    all_pairs = [f'{o}_{op}' for o in orphan_list for op in opioid_list]
    ready_pairs = [p for p in all_pairs if os.path.exists(f'{WORK_DIR}/{p}.prmtop')]
    if not ready_pairs:
        print('No systems ready for production')
    else:
        gpu0_pairs = ready_pairs[:3]
        gpu1_pairs = ready_pairs[3:]
        print(f'GPU 0 will run: {gpu0_pairs}')
        print(f'GPU 1 will run: {gpu1_pairs}')
        def gpu_worker(pairs, g_idx):
            for p in pairs:
                print(f'\n--- Starting production for {p} on GPU {g_idx} ---')
                r = subprocess.run([sys.executable, MD_ENGINE, CONFIG_PATH, 'prod', p, WORK_DIR, str(g_idx)],
                                   capture_output=False, text=True)
                if r.returncode != 0:
                    print(f'ERROR: production for {p} failed')
        threads = []
        for gpu_idx, gpu_pairs in [(0, gpu0_pairs), (1, gpu1_pairs)]:
            if gpu_pairs:
                t = threading.Thread(target=gpu_worker, args=(gpu_pairs, gpu_idx))
                t.start()
                threads.append(t)
        for t in threads:
            t.join()
        print('All production runs completed')
else:
    print('Skipping production')


# 5. Run equilibration for all 6 pairs via subprocess (conda Python with CUDA)
import subprocess, os, sys
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
MD_ENGINE = f'{WORK_DIR}/md_engine.py'
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
import os
if RUN_STAGE['equil']:
    gpu_idx = 0
    for orphan in orphan_list:
        for opioid in opioid_list:
            pair = f'{orphan}_{opioid}'
            if os.path.exists(f'{WORK_DIR}/{pair}.prmtop'):
                print(f'\n--- Running equilibration for {pair} on GPU {gpu_idx} ---')
                my_env = os.environ.copy()
                my_env['LD_LIBRARY_PATH'] = f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + my_env.get('LD_LIBRARY_PATH', '')
                r = subprocess.run([OPENMM_PYTHON, MD_ENGINE, CONFIG_PATH, 'equil', pair, WORK_DIR, str(gpu_idx)],
                                   capture_output=False, text=True, env=my_env)
                if r.returncode != 0:
                    print(f'ERROR: equilibration for {pair} failed')
                gpu_idx = 1 - gpu_idx
            else:
                print(f'{pair}: missing prmtop, skipping')
else:
    print('Skipping equilibration')


In [ ]:
# 7a. Analysis functions (load one trajectory at a time)
import json
import os
import MDAnalysis as mda
from MDAnalysis.analysis import distances, align
import numpy as np
def analyze_pair(pair_name, work_dir):
    pdb = f'{work_dir}/{pair_name}_equil.pdb'
    dcd = f'{work_dir}/{pair_name}_prod.dcd'
    if not os.path.exists(dcd):
        print(f'{pair_name}: no trajectory')
        return None
    u = mda.Universe(pdb, dcd)
    split_file = f'{work_dir}/{pair_name}_split.json'
    if os.path.exists(split_file):
        with open(split_file) as jf:
            split = json.load(jf)
        n_A, n_B = split['n_A'], split['n_B']
    else:
        u_complex = mda.Universe(f'{work_dir}/{pair_name}_complex.pdb')
        n_A = len(u_complex.select_atoms('chainid A').residues)
        n_B = len(u_complex.select_atoms('chainid B').residues)
    ca_A = u.select_atoms(f'resid 1-{n_A} and name CA')
    ca_B = u.select_atoms(f'resid {n_A+1}-{n_A+n_B} and name CA')
    all_A = u.select_atoms(f'resid 1-{n_A}')
    all_B = u.select_atoms(f'resid {n_A+1}-{n_A+n_B}')
    com_dists, iface_rmsd, contact_counts, hbond_proxy = [], [], [], []
    u.trajectory[0]
    ref_A = ca_A.positions.copy()
    ref_B = ca_B.positions.copy()
    for ts in u.trajectory:
        com_dists.append(np.linalg.norm(all_A.center_of_mass() - all_B.center_of_mass()))
        R, _ = align.rotation_matrix(ca_A.positions, ref_A)
        aligned_B = np.dot(ca_B.positions - ca_B.positions.mean(axis=0), R) + ref_B.mean(axis=0)
        iface_rmsd.append(np.sqrt(np.mean(np.sum((aligned_B - ref_B)**2, axis=1))))
        dist_mat = distances.distance_array(all_A.positions, all_B.positions)
        contact_counts.append(np.sum(dist_mat < 5.0))
        if ts.frame % 10 == 0:
            donors = all_A.select_atoms('name N* or name O*')
            acceptors = all_B.select_atoms('name N* or name O*')
            d = distances.distance_array(donors.positions, acceptors.positions)
            hbond_proxy.append(np.sum(d < 3.5))
    return {'com': np.array(com_dists), 'rmsd': np.array(iface_rmsd),
            'contacts': np.array(contact_counts), 'hbonds': np.array(hbond_proxy)}


In [ ]:
# 6. Run production CONCURRENTLY on 2 GPUs via subprocess (conda Python with CUDA)
import threading, subprocess, os
OPENMM_ENV = '/opt/miniconda/envs/openmm_env'
OPENMM_PYTHON = f'{OPENMM_ENV}/bin/python'
MD_ENGINE = f'{WORK_DIR}/md_engine.py'
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
if RUN_STAGE['production']:
    all_pairs = [f'{o}_{op}' for o in orphan_list for op in opioid_list]
    ready_pairs = [p for p in all_pairs if os.path.exists(f'{WORK_DIR}/{p}.prmtop')]
    if not ready_pairs:
        print('No systems ready for production')
    else:
        gpu0_pairs = ready_pairs[:3]
        gpu1_pairs = ready_pairs[3:]
        print(f'GPU 0 will run: {gpu0_pairs}')
        print(f'GPU 1 will run: {gpu1_pairs}')
        def gpu_worker(pairs, g_idx):
            for p in pairs:
                print(f'\n--- Starting production for {p} on GPU {g_idx} ---')
                my_env = os.environ.copy()
                my_env['LD_LIBRARY_PATH'] = f'{OPENMM_ENV}/lib:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:' + my_env.get('LD_LIBRARY_PATH', '')
                r = subprocess.run([OPENMM_PYTHON, MD_ENGINE, CONFIG_PATH, 'prod', p, WORK_DIR, str(g_idx)],
                                   capture_output=False, text=True, env=my_env)
                if r.returncode != 0:
                    print(f'ERROR: production for {p} failed')
        threads = []
        for gpu_idx, gpu_pairs in [(0, gpu0_pairs), (1, gpu1_pairs)]:
            if gpu_pairs:
                t = threading.Thread(target=gpu_worker, args=(gpu_pairs, gpu_idx))
                t.start()
                threads.append(t)
        for t in threads:
            t.join()
        print('All production runs completed')
else:
    print('Skipping production')


In [ ]:
# 7c. 6-system comparison dashboard
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('GPR151 vs GPR88: Opioid Heterodimerization Analysis', fontsize=16)
colors = {
    'gpr151_mor': 'crimson', 'gpr151_kor': 'orangered', 'gpr151_dor': 'darkred',
    'gpr88_mor': 'steelblue', 'gpr88_kor': 'royalblue', 'gpr88_dor': 'navy'
}
for pair, data in results.items():
    c = colors.get(pair, 'gray')
    axes[0,0].plot(data['com'], color=c, alpha=0.7, label=pair)
    axes[0,1].plot(data['rmsd'], color=c, alpha=0.7, label=pair)
    axes[0,2].plot(data['contacts'], color=c, alpha=0.7, label=pair)
    x = np.linspace(0, len(data['contacts'])-1, len(data['hbonds']))
    axes[1,0].plot(x, data['hbonds'], color=c, alpha=0.7, label=pair)
    axes[1,1].hist(data['com'], bins=30, alpha=0.4, color=c, density=True)
axes[0,0].set_title('1. COM Distance'); axes[0,0].set_ylabel('COM (A)'); axes[0,0].legend(fontsize=7)
axes[0,1].set_title('2. Interface RMSD (aligned)'); axes[0,1].set_ylabel('RMSD (A)'); axes[0,1].legend(fontsize=7)
axes[0,1].axhline(y=2.5, color='k', linestyle='--', alpha=0.3)
axes[0,2].set_title('3. Contact Count'); axes[0,2].set_ylabel('Contacts (<5A)'); axes[0,2].legend(fontsize=7)
axes[1,0].set_title('4. H-bond Proxy'); axes[1,0].set_ylabel('H-bonds'); axes[1,0].legend(fontsize=7)
axes[1,1].set_title('5. COM Distribution'); axes[1,1].set_xlabel('COM (A)'); axes[1,1].set_ylabel('Density')
ax = axes[1,2]
pairs = sorted(results.keys())
x_pos = np.arange(len(pairs))
com_means = [results[p]['com'].mean() for p in pairs]
rmsd_means = [results[p]['rmsd'].mean() for p in pairs]
bar_colors = [colors.get(p, 'gray') for p in pairs]
ax.bar(x_pos - 0.2, com_means, 0.4, label='COM mean', color=bar_colors, alpha=0.7)
ax.bar(x_pos + 0.2, rmsd_means, 0.4, label='RMSD mean', color=bar_colors, alpha=0.4, edgecolor='k')
ax.set_xticks(x_pos); ax.set_xticklabels(pairs, rotation=45, ha='right', fontsize=8)
ax.set_title('6. Summary'); ax.legend()
plt.tight_layout()
out = f'{WORK_DIR}/heterodimerization_dashboard.png'
plt.savefig(out, dpi=300, bbox_inches='tight')
print(f'Saved: {out}')
plt.show()


## SECTION 8: Binary Conclusion & Evidence Summary


In [ ]:
# 8a. Score each pair and compare GPR151 to GPR88 positive controls
print('='*70)
print('HETERODIMERIZATION EVIDENCE SUMMARY')
print('='*70)
scores = {}
for pair, data in results.items():
    n = len(data['com'])
    com_late = data['com'][n//2:]
    rmsd_late = data['rmsd'][n//2:]
    contacts_late = data['contacts'][n//2:]
    s = {}
    s['com_stable'] = com_late.std() < 5.0
    s['com_in_range'] = 25 < com_late.mean() < 45
    s['rmsd_low'] = rmsd_late.mean() < 3.0
    s['contacts_persistent'] = contacts_late.mean() > 50
    s['no_drift'] = abs(com_late[-1] - com_late[0]) < 10
    s['total'] = sum(s.values())
    scores[pair] = s
    print(f"\n{pair}:")
    for k,v in s.items():
        print(f'  {k}: {"PASS" if v else "FAIL"}')
    print(f'  Score: {s["total"]}/5')
print('\n' + '='*70)
print('CROSS-VALIDATION: GPR151 vs GPR88 per opioid receptor')
print('='*70)
for opioid in opioid_list:
    s151 = scores.get(f'gpr151_{opioid}', {}).get('total', 0)
    s88 = scores.get(f'gpr88_{opioid}', {}).get('total', 0)
    print(f"\n{opioid.upper()}:")
    print(f'  GPR151: {s151}/5')
    print(f'  GPR88:  {s88}/5')
    if s151 >= 4 and s88 >= 4:
        print('  -> GPR151 STRONGLY supports heterodimerization (comparable to GPR88)')
    elif s151 >= 3 and s88 >= 4:
        print('  -> GPR151 shows MODERATE evidence (weaker than GPR88)')
    elif s151 < 3 and s88 >= 4:
        print('  -> GPR151 DOES NOT support stable heterodimerization (unlike GPR88)')
        print('  -> Aligns with Yoshida et al. (2025) experimental null')
    elif s151 < 3 and s88 < 3:
        print('  -> Neither shows strong evidence for this opioid receptor')
print('\n' + '='*70)
print('OVERALL')
print('='*70)
total_151 = sum(scores.get(f'gpr151_{o}', {}).get('total', 0) for o in opioid_list)
total_88 = sum(scores.get(f'gpr88_{o}', {}).get('total', 0) for o in opioid_list)
print(f'GPR151 total: {total_151}/15')
print(f'GPR88 total:  {total_88}/15')
if total_88 == 0 and total_151 == 0:
    print('\nNo results available. Run production MD first.')
elif total_151 >= total_88 * 0.8:
    print('\nGPR151 shows comparable heterodimerization propensity to GPR88.')
else:
    print('\nGPR151 shows significantly weaker heterodimerization than GPR88.')


## PHASE 2: Extend Hits to 100 ns


In [ ]:
# PHASE 2: Extend hits to 100 ns
HIT_PAIRS = []
MD_ENGINE = f'{WORK_DIR}/md_engine.py'
CONFIG_PATH = f'{WORK_DIR}/md_config.json'
if not HIT_PAIRS:
    print('No HIT_PAIRS selected. Inspect the Phase 1 dashboard and set HIT_PAIRS.')
else:
    N_STEPS_TOTAL = int(100 * 1000 / STEP_SIZE)
    print(f'Phase 2: extending {HIT_PAIRS} to 100 ns each')
    print(f'N_STEPS_TOTAL = {N_STEPS_TOTAL} ({N_STEPS_TOTAL * STEP_SIZE / 1000:.1f} ns)')
    import json
    with open(CONFIG_PATH) as f:
        cfg = json.load(f)
    cfg['N_STEPS_TOTAL'] = N_STEPS_TOTAL
    with open(CONFIG_PATH, 'w') as f:
        json.dump(cfg, f)
    gpu_idx = 0
    for pair in HIT_PAIRS:
        if os.path.exists(f'{WORK_DIR}/{pair}.prmtop'):
            print(f'\n--- Starting Phase 2 production for {pair} on GPU {gpu_idx} ---')
            r = subprocess.run([sys.executable, MD_ENGINE, CONFIG_PATH, 'prod', pair, WORK_DIR, str(gpu_idx)],
                               capture_output=False, text=True)
            if r.returncode != 0:
                print(f'ERROR: Phase 2 production for {pair} failed')
            gpu_idx = 1 - gpu_idx
        else:
            print(f'{pair}: missing prmtop, skipping')
    print('Phase 2 complete. Re-run analysis cells to see 100 ns results.')
